In [ ]:
!pip install torch torchaudio
!pip install funasr
!pip install modelscope

drive_model_path = "/content/drive/MyDrive/modelscope_cache"
asr_model = "speech_seaco_paraformer_large_asr_nat-zh-cn-16k-common-vocab8404-pytorch" #SeACoParaformer-zh
vad_model = "speech_fsmn_vad_zh-cn-16k-common-pytorch" #fsmn-vad
punc_model = "punc_ct-transformer_cn-en-common-vocab471067-large" #ct-punc

!modelscope download --model iic/{asr_model} --local_dir {drive_model_path}/iic/{asr_model}
!modelscope download --model iic/{vad_model} --local_dir {drive_model_path}/iic/{vad_model}
!modelscope download --model iic/{punc_model} --local_dir {drive_model_path}/iic/{punc_model}

import yaml
import shutil
import os

%env TZ=Asia/Shanghai

#环境变量
PROJECT_DIR = '/content/drive/MyDrive/github/bilibili2txt'
CONFIG_PATH = f'{PROJECT_DIR}/config.yaml'

#加载配置
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config_data = yaml.safe_load(f) or {}

server_cfg = config_data.get('server', {})
server_id = server_cfg.get('server_id', 'colab-server')
git_email = server_cfg.get('git_email') or f'{server_id}@users.noreply.github.com'
ssh_key_path = str(server_cfg.get('ssh_key_path') or '').strip()
git_ssh_user = str(server_cfg.get('git_ssh_user') or 'git').strip()

#配置github
!git config --global user.name {server_id}
!git config --global user.email {git_email}

#配置证书（在 config.yaml 的 server.ssh_key_path 中配置）
if ssh_key_path:
    ssh_dir = os.path.expanduser('~/.ssh')
    os.makedirs(ssh_dir, exist_ok=True)
    dst_key = os.path.join(ssh_dir, 'id_ed25519')
    shutil.copy2(ssh_key_path, dst_key)
    os.chmod(dst_key, 0o600)
    with open(os.path.join(ssh_dir, 'config'), 'w', encoding='utf-8') as f:
        f.write(
            "Host github.com\n"
            "  HostName github.com\n"
            f"  User {git_ssh_user}\n"
            "  IdentityFile ~/.ssh/id_ed25519\n"
            "  StrictHostKeyChecking accept-new\n"
        )

#whisper改成可运行
faster_whisper_path = server_cfg.get('faster_whisper_path', '/content/drive/MyDrive/Faster-Whisper-XXL/faster-whisper-xxl')
if faster_whisper_path and os.path.exists(faster_whisper_path):
    os.chmod(faster_whisper_path, 0o755)

#安装软件
!apt-get update -y
!apt-get install -y ffmpeg git

#升级，并且，安装python库
%cd {PROJECT_DIR}
!git pull
!python -m pip install -U pip
!python -m pip install -r requirements.txt

#清空queue
queue_cfg = config_data.get('queue', {})
queue_repo_dir = queue_cfg.get('repo_dir', 'queue')
queue_repo_url = str(queue_cfg.get('repo_url') or '').strip()
if not queue_repo_url or '您的用户名' in queue_repo_url or 'your-username' in queue_repo_url:
    raise ValueError('请在 config.yaml 中配置有效的 queue.repo_url')
if not os.path.isabs(queue_repo_dir):
    queue_repo_dir = os.path.abspath(os.path.join(PROJECT_DIR, queue_repo_dir))
print("解析后的 queue 绝对路径:", queue_repo_dir)
!rm -rf {queue_repo_dir}
!git clone {queue_repo_url} {queue_repo_dir}

#运行
!python b2t.py --config {CONFIG_PATH} server run